# 04 — Backtesting

Run the full Nexus Trader agent pipeline against historical data.

**Topics covered:**
- Walk-forward simulation using DuckDB history
- Per-trade P&L tracking with R:R enforcement
- Circuit-breaker impact analysis
- Session-segmented performance
- Equity curve, drawdown, and QuantStats tear sheet

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DB_PATH = "../data/nexus.duckdb"

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
trades = conn.execute("SELECT * FROM trades ORDER BY opened_at").fetchdf()
conn.close()
print(f"Total trades: {len(trades)}")
trades.head()

In [ ]:
# Equity curve
if not trades.empty and 'pnl_usd' in trades.columns:
    trades['cumulative_pnl'] = trades['pnl_usd'].cumsum()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=trades['closed_at'], y=trades['cumulative_pnl'], mode='lines', name='Equity'))
    fig.update_layout(title='Equity Curve', xaxis_title='Date', yaxis_title='Cumulative P&L ($)')
    fig.show()

In [ ]:
# Drawdown analysis
if not trades.empty and 'cumulative_pnl' in trades.columns:
    peak = trades['cumulative_pnl'].expanding().max()
    drawdown = trades['cumulative_pnl'] - peak
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=trades['closed_at'], y=drawdown, fill='tozeroy', name='Drawdown'))
    fig.update_layout(title='Drawdown', xaxis_title='Date', yaxis_title='Drawdown ($)')
    fig.show()
    print(f"Max Drawdown: ${drawdown.min():.2f}")

In [ ]:
# Session-segmented performance
if not trades.empty and 'session' in trades.columns:
    session_stats = trades.groupby('session').agg(
        count=('pnl_usd', 'count'),
        total_pnl=('pnl_usd', 'sum'),
        avg_pnl=('pnl_usd', 'mean'),
        win_rate=('pnl_usd', lambda x: (x > 0).mean()),
    ).round(2)
    display(session_stats)

In [ ]:
# Key metrics
if not trades.empty:
    wins = trades[trades['pnl_usd'] > 0]
    losses = trades[trades['pnl_usd'] <= 0]
    total_pnl = trades['pnl_usd'].sum()
    win_rate = len(wins) / len(trades) if len(trades) else 0
    profit_factor = wins['pnl_usd'].sum() / abs(losses['pnl_usd'].sum()) if len(losses) and losses['pnl_usd'].sum() != 0 else float('inf')
    avg_rr = trades['rr_ratio'].mean() if 'rr_ratio' in trades.columns else 0

    print(f"Total P&L:     ${total_pnl:.2f}")
    print(f"Win Rate:      {win_rate:.1%}")
    print(f"Profit Factor: {profit_factor:.2f}")
    print(f"Avg R:R:       {avg_rr:.2f}")
    print(f"Total Trades:  {len(trades)}")